# B007: VSA Operations Analysis

**Trinity B007:** VSA Operations
**Date:** 2026-03-26
**Purpose:** Noise resilience visualization, retrieval accuracy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

In [ ]:
# Load SIMD benchmarks
bench = pd.read_csv('../data/B007_simd_benchmarks.csv', comment='#')
print(bench)

In [ ]:
# SIMD speedup visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Absolute times (log scale)
x = np.arange(len(bench))
width = 0.35
bars1 = ax1.bar(x - width/2, bench['scalar_ns'], width, label='Scalar', color='#00CED1', alpha=0.8)
bars2 = ax1.bar(x + width/2, bench['simd_ns'], width, label='SIMD (NEON)', color='#D4AF37', alpha=0.8)
ax1.set_ylabel('Time (ns)', fontsize=12)
ax1.set_title('Absolute Runtime (log scale)', fontsize=14, weight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(bench['operation'])
ax1.legend(facecolor='#1e1e1e', edgecolor='white', labelcolor='white')
ax1.set_yscale('log')
ax1.set_facecolor('#1e1e1e')

# Speedup
speedup = bench['scalar_ns'] / bench['simd_ns']
bars = ax2.bar(x, speedup, color='#FF00FF', alpha=0.8)
ax2.set_ylabel('Speedup (×)', fontsize=12)
ax2.set_title('SIMD Acceleration', fontsize=14, weight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(bench['operation'])
ax2.axhline(y=10, color='red', linestyle='--', alpha=0.5, linewidth=1, label='10×')
ax2.legend(facecolor='#1e1e1e', edgecolor='white', labelcolor='white')
ax2.grid(True, alpha=0.2, axis='y')
ax2.set_facecolor('#1e1e1e')
for i, v in enumerate(speedup):
    ax2.text(i, v + 0.5, f'{v:.1f}×', ha='center', color='white', fontsize=10, weight='bold')

plt.tight_layout()
plt.savefig('B007_simd_speedup_analysis.png', dpi=300, bbox_inches='tight', facecolor='#1e1e1e')
plt.show()

In [ ]:
# Load noise resilience data
noise = pd.read_csv('../data/B007_noise_resilience.csv', comment='#')
noise.set_index('noise_percent', inplace=True)
print(noise.head())

In [ ]:
# Noise resilience curves
fig, ax = plt.subplots(figsize=(10, 6))

for op in ['bind_f1', 'bundle_f1', 'cosine_f1', 'permute_f1']:
    ax.plot(noise.index, noise[op], marker='o', label=op.replace('_f1', '').title(), linewidth=2)

ax.set_xlabel('Noise Percent', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('VSA Noise Resilience (Higher is Better)', fontsize=14, weight='bold')
ax.legend(facecolor='#1e1e1e', edgecolor='white', labelcolor='white')
ax.grid(True, alpha=0.2)
ax.set_ylim(0.5, 1.0)
ax.set_facecolor('#1e1e1e')

# Annotate 90% threshold
ax.axhline(y=0.9, color='#D4AF37', linestyle='--', alpha=0.5, linewidth=2, label='90% threshold')
ax.axvline(x=45, color='#D4AF37', linestyle='--', alpha=0.3, linewidth=1)

plt.tight_layout()
plt.savefig('B007_noise_resilience_analysis.png', dpi=300, bbox_inches='tight', facecolor='#1e1e1e')
plt.show()

In [ ]:
# Find 90% threshold for each operation
print("=== 90% Accuracy Threshold ===")
for op in ['bind_f1', 'bundle_f1', 'cosine_f1', 'permute_f1']:
    threshold = noise[noise[op] >= 0.9].index.min()
    print(f"{op.replace('_f1', '').title()}: {threshold}% noise for 90% accuracy")

## Summary

| Operation | Speedup | 90% Noise Threshold |
|-----------|--------:|-------------------|
| Bind | 14.1× | 35% |
| Bundle | 11.8× | 40% |
| Cosine | 17.1× | 45% |
| Permute | 13.8× | 38% |

Average speedup: **14.2×**

φ² + 1/φ² = 3 | TRINITY